# Modelo de Incêndio Florestal (Fire Model)
### Simulação de propagação de fogo com dissmodel

Este modelo simula a dinâmica de propagação de um incêndio em uma floresta homogênea. É um exemplo clássico de percolação em grades, onde o estado de uma célula depende da presença de vizinhos em combustão.

## Conceitos Fundamentais

Cada célula pode estar em um de quatro estados:

| Estado | Descrição | Valor |
|:---:|:---|:---:|
| **Floresta** | Árvores saudáveis que podem pegar fogo. | 0 |
| **Em chamas** | Árvores queimando ativamente, propagando fogo. | 1 |
| **Queimado** | Área já consumida pelo fogo, não queima mais. | 2 |
| **Vazio** | Sem vegetação; nunca pega fogo. | 3 |

**Regras de Transição (Vizinhança de Rook/4 direções):**

1. Uma célula **Em chamas** sempre se torna **Queimada** no próximo passo.
2. Uma célula de **Floresta** torna-se **Em chamas** se pelo menos um de seus vizinhos imediatos estiver queimando.
3. Células **Queimadas** e **Vazias** não mudam de estado.

## Imports

In [ ]:
from matplotlib.colors import ListedColormap

from dissmodel.core import Environment
from dissmodel.geo import vector_grid
from dissmodel.visualization.map import Map
from dissmodel_ca.models import FireModel

## ⚠️ Ordem correta de inicialização

O `Environment` deve ser criado primeiro, seguido pelo modelo e pela visualização.

```
Environment  →  FireModel  →  Map
```

In [ ]:
env = Environment(start_time=0, end_time=20)

## Construção do espaço/modelo

Criamos uma grade de 25x25 representando nossa floresta.

In [ ]:
gdf = vector_grid(dimension=(25, 25), resolution=1, attrs={"state": 0})
gdf.head()

## Inicialização

Por padrão, a inicialização segue o modelo `Fire` do pacote `ca` do TerraME: cada célula fica vazia com probabilidade `empty` (10%) e é floresta nos outros casos, e uma única célula, sorteada entre todas, começa em chamas. Com `initial_fire_density` maior que zero, cada célula começa em chamas com essa probabilidade, no lugar do foco único.

In [ ]:
model = FireModel(gdf=gdf)
model.setup(empty=0.1, initial_fire_density=0.02)  # 10% vazio; 2% das células começam pegando fogo
model.initialize()

## Visualização

Cores sugeridas: Verde (Floresta), Vermelho (Em chamas), Cinza (Queimado), Branco (Vazio). `vmin` e `vmax` fixam a escala nos quatro estados.

In [ ]:
cmap = ListedColormap(["#27ae60", "#e74c3c", "#7f8c8d", "#ffffff"])
Map(
    gdf=gdf,
    plot_params={"column": "state", "cmap": cmap, "vmin": 0, "vmax": 3, "ec": "#2ecc71", "lw": 0.5},
)

## Execução da simulação

In [ ]:
env.run()

## Exploração de parâmetros / Experimentos guiados

- **Foco único (como no TerraME):** use os valores padrão, que deixam 10% das células vazias e sorteiam um único foco:
  ```python
  model.setup()
  model.initialize()
  ```
- **Limiar de percolação:** Aumente `empty` (por exemplo, 0.3, 0.4, 0.5). A partir de que proporção de células vazias o fogo deixa de atravessar a floresta?
- **Densidade Crítica:** Como a `initial_fire_density` afeta a proporção final da floresta queimada?
- **Extensão:** Aumente o `end_time` para garantir que todo o fogo se extinga (quando não houver mais células no estado 1).

## Conclusão

Este modelo demonstra como um processo local de contágio pode levar à exaustão de recursos (neste caso, as árvores). É um modelo fundamental para entender fenômenos de propagação em redes e sistemas espaciais.